In [2]:
import os
import json
import urllib.parse
import requests
from datetime import date
from astropy.time import Time

FRITZ_TOKEN = os.environ.get("FRITZ_FOLLOWUP_REQ_TOKEN")
FRITZ_BASE = "https://fritz.science"

def fritz_api(endpoint, params=None):
    headers = {"Authorization": f"token {FRITZ_TOKEN}"}
    url = urllib.parse.urljoin(FRITZ_BASE, endpoint)
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

# ---------------------------------------------------------------------------
# fetch all runs and filter for upcoming SOAR runs
# ---------------------------------------------------------------------------

SOAR_INSTRUMENT_IDS = {1107, 1108, 1109}

r_all = fritz_api("/api/observing_run")
all_runs = r_all["data"]

today = date.today().isoformat()
now_jd = Time.now().jd

soar_runs = [
    run for run in all_runs
    if run.get("instrument_id") in SOAR_INSTRUMENT_IDS
]

upcoming_soar_runs = sorted(
    [run for run in soar_runs if run["calendar_date"] >= today],
    key=lambda r: r["calendar_date"]
)

# compute dt in JD for each run (observation starts at 19:00 UTC = 3pm EDT)
relevant_runs = []
for run in upcoming_soar_runs:
    run_start_jd = Time(f"{run['calendar_date']}T19:00:00", format="isot", scale="utc").jd
    dt = run_start_jd - now_jd
    if dt <= 2.0:
        relevant_runs.append((run, dt))

# print
if not relevant_runs:
    print("No upcoming SOAR runs within the next 2 days.")
else:
    if len(relevant_runs) >= 2:
        print("⚠️  2 back-to-back observing nights found!\n")
    for run, dt in relevant_runs:
        label = "🔴 TONIGHT" if dt <= 1.0 else "TOMORROW"
        print(f"{label}  |  Run ID: {run['id']}  |  Date: {run['calendar_date']}  |  PI: {run['pi']}  |  dt: {dt:.3f} JD")

TOMORROW  |  Run ID: 2202  |  Date: 2026-06-13  |  PI: Andreoni, Carney  |  dt: 1.136 JD


In [6]:
all_assignments = []
for run, dt in relevant_runs:
    r = fritz_api(f"/api/observing_run/{run['id']}")
    assignments = r["data"].get("assignments", [])
    all_assignments.append((run, dt, assignments))
    
    label = "🔴 TONIGHT" if dt <= 1.0 else "TOMORROW"
    print(f"\n{'='*80}")
    print(f"{label}  |  Run ID: {run['id']}  |  Date: {run['calendar_date']}  |  PI: {run['pi']}")
    print(f"{'='*80}")


    
    if not assignments:
        print("  No targets assigned so far.")
    else:
        print(f"  Total targets: {len(assignments)}")
        for n, a in enumerate(assignments):
            print(f"{n+1}. {a['requester']['username']} has requested https://fritz.science/source/{a['obj_id']} with priority {a['priority']} ({a['status']})")


TOMORROW  |  Run ID: 2202  |  Date: 2026-06-13  |  PI: Andreoni, Carney
  Total targets: 1
1. ariel has requested https://fritz.science/source/ZTF26aaxhpdm with priority 3 (pending)


In [8]:
obj_ids = list({
    a["obj_id"]
    for run, dt, assignments in all_assignments
    for a in assignments
})

print(f"Unique targets: {len(obj_ids)}")
print(obj_ids)

Unique targets: 1
['ZTF26aaxhpdm']


In [13]:
from datetime import timedelta

LCO_TOKEN = os.environ.get("LCO_TOKEN") #or "YOUR_LCO_TOKEN_HERE"
LCO_BASE = "https://observe.lco.global"

def lco_api(endpoint, params=None):
    headers = {"Authorization": f"Token {LCO_TOKEN}"}
    url = urllib.parse.urljoin(LCO_BASE, endpoint)
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

# test with the one target
run, dt, assignments = all_assignments[0]
run_date = date.fromisoformat(run["calendar_date"])
date_start = (run_date - timedelta(days=3)).isoformat()
date_end = (run_date + timedelta(days=3)).isoformat()

obj_id = assignments[0]["obj_id"]

r = lco_api("/api/requestgroups/", params={
    "target_name": obj_id,
    "created_after": f"{date_start}T00:00:00Z",
    "created_before": f"{date_end}T00:00:00Z",
    "limit": 100,
})

print(f"Target: {obj_id}  |  Window: {date_start} to {date_end}  |  LCO matches: {r['count']}")
for rg in r["results"]:
    target_name = rg["requests"][0]["configurations"][0]["target"]["name"]
    if target_name == obj_id:
        print(f"  LCO ID: {rg['id']}  |  Target: {target_name}  |  State: {rg['state']}  |  Proposal: {rg['proposal']}  |  Submitter: {rg['submitter']}  |  Created: {rg['created'][:10]}")

Target: ZTF26aaxhpdm  |  Window: 2026-06-10 to 2026-06-16  |  LCO matches: 4
  LCO ID: 2533292  |  Target: ZTF26aaxhpdm  |  State: CANCELED  |  Proposal: SOAR2026A-021  |  Submitter: jcarney  |  Created: 2026-06-11


In [14]:
for run, dt, assignments in all_assignments:
    run_date = date.fromisoformat(run["calendar_date"])
    date_start = (run_date - timedelta(days=3)).isoformat()
    date_end = (run_date + timedelta(days=3)).isoformat()
    
    label = "🔴 TONIGHT" if dt <= 1.0 else "TOMORROW"
    print(f"\n{'='*80}")
    print(f"{label}  |  Run ID: {run['id']}  |  Date: {run['calendar_date']}  |  PI: {run['pi']}")
    print(f"{'='*80}")
    
    if not assignments:
        print("  No targets assigned so far.")
        continue
    
    for a in assignments:
        obj_id = a["obj_id"]
        print(f"\n  Target:    {obj_id}")
        print(f"  Requester: {a['requester']['username']}")
        print(f"  Priority:  {a['priority']}")
        print(f"  Status:    {a['status']}")
        print(f"  Comment:   {a.get('comment', 'N/A')}")
        
        r = lco_api("/api/requestgroups/", params={
            "target_name": obj_id,
            "created_after": f"{date_start}T00:00:00Z",
            "created_before": f"{date_end}T00:00:00Z",
            "limit": 100,
        })
        
        lco_matches = [
            rg for rg in r["results"]
            if rg["requests"][0]["configurations"][0]["target"]["name"] == obj_id
        ]
        
        if not lco_matches:
            print(f"  LCO:       NOT FOUND in ±3 day window")
        else:
            for rg in lco_matches:
                print(f"  LCO ID: {rg['id']}  |  State: {rg['state']}  |  Proposal: {rg['proposal']}  |  Submitter: {rg['submitter']}  |  Created: {rg['created'][:10]}")


TOMORROW  |  Run ID: 2202  |  Date: 2026-06-13  |  PI: Andreoni, Carney

  Target:    ZTF26aaxhpdm
  Requester: ariel
  Priority:  3
  Status:    pending
  Comment:   Lensec SN
  LCO ID: 2533292  |  State: CANCELED  |  Proposal: SOAR2026A-021  |  Submitter: jcarney  |  Created: 2026-06-11


In [10]:
for rg in r["results"]:
    print(f"  LCO ID: {rg['id']}  |  State: {rg['state']}  |  Proposal: {rg['proposal']}  |  Submitter: {rg['submitter']}  |  Created: {rg['created'][:10]}")

  LCO ID: 2533292  |  State: CANCELED  |  Proposal: SOAR2026A-021  |  Submitter: jcarney  |  Created: 2026-06-11
  LCO ID: 2532513  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10
  LCO ID: 2532512  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10
  LCO ID: 2532511  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10


In [11]:
for rg in r["results"]:
    target_name = rg["requests"][0]["configurations"][0]["target"]["name"]
    print(f"  LCO ID: {rg['id']}  |  Target: {target_name}  |  State: {rg['state']}  |  Proposal: {rg['proposal']}  |  Submitter: {rg['submitter']}  |  Created: {rg['created'][:10]}")

  LCO ID: 2533292  |  Target: ZTF26aaxhpdm  |  State: CANCELED  |  Proposal: SOAR2026A-021  |  Submitter: jcarney  |  Created: 2026-06-11
  LCO ID: 2532513  |  Target: VAST_04  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10
  LCO ID: 2532512  |  Target: VAST_04  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10
  LCO ID: 2532511  |  Target: VAST_04  |  State: PENDING  |  Proposal: SOAR2026A-018  |  Submitter: jimfreeburn  |  Created: 2026-06-10


In [12]:
for rg in r["results"]:
    target_name = rg["requests"][0]["configurations"][0]["target"]["name"]
    if target_name == obj_id:
        print(f"  LCO ID: {rg['id']}  |  Target: {target_name}  |  State: {rg['state']}  |  Proposal: {rg['proposal']}  |  Submitter: {rg['submitter']}  |  Created: {rg['created'][:10]}")

  LCO ID: 2533292  |  Target: ZTF26aaxhpdm  |  State: CANCELED  |  Proposal: SOAR2026A-021  |  Submitter: jcarney  |  Created: 2026-06-11
